# Midterm-Style Submission Notebook

This is the only supported submission notebook.
It mirrors the simple one-pass flow from `DL_Midterm_Eval.ipynb`, but uses the current Google notebook setup for Drive mount, repo checkout, merged-model discovery, and persistent output files.

Historical experiment notebooks are preserved in `archive/` for documentation, but they are not supported execution paths.

This next cell mounts Google Drive, clones the repo into the runtime, copies the required CSV inputs into the checkout, and defines the persistent Drive output directory.

- Reads: Google Drive, GitHub repo, runtime filesystem
- Writes: `/content/svg-kaggle-comptetition`, Drive output folders
- Rerun-safe: Yes. It resets the runtime checkout and recopies the inputs.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

GIT_REPO_URL = "https://github.com/Demetri65/svg-kaggle-competition-.git"
CHECKOUT_PATH = Path("/content/svg-kaggle-comptetition")
DRIVE_ROOT = Path("/content/drive/MyDrive")
PROJECT_ROOT_ON_DRIVE = DRIVE_ROOT / "svg-kaggle-comptetition"
OUTPUT_ROOT = PROJECT_ROOT_ON_DRIVE / "submission_outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if CHECKOUT_PATH.exists():
    shutil.rmtree(CHECKOUT_PATH)
subprocess.check_call(["git", "clone", GIT_REPO_URL, str(CHECKOUT_PATH)])

for required_name in ["test.csv", "train.csv", "sample_submission.csv"]:
    destination = CHECKOUT_PATH / required_name
    candidate_paths = [
        PROJECT_ROOT_ON_DRIVE / required_name,
        DRIVE_ROOT / "Colab Notebooks" / "svg-kaggle-comptetition" / required_name,
    ]
    for candidate in candidate_paths:
        if candidate.exists():
            shutil.copy2(candidate, destination)
            break
    else:
        raise FileNotFoundError(f"Could not find {required_name} in Google Drive.")

os.chdir(CHECKOUT_PATH)
print(f"Repo checkout: {CHECKOUT_PATH}")
print(f"Persistent output root: {OUTPUT_ROOT}")


## Package Check

This next cell installs the runtime packages needed by the midterm-style notebook, using normal Python subprocess calls instead of notebook shell magics.

- Reads: Current Python environment
- Writes: Installed runtime packages
- Rerun-safe: Yes. It only installs packages that are missing.


In [ ]:
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {
    "transformers": "transformers",
    "peft": "peft",
    "accelerate": "accelerate",
    "bitsandbytes": "bitsandbytes",
    "datasets": "datasets",
    "trl": "trl",
    "cairosvg": "cairosvg",
    "lxml": "lxml",
    "pandas": "pandas",
    "numpy": "numpy",
    "tqdm": "tqdm",
    "PIL": "pillow",
}

missing_packages = [
    package_name
    for module_name, package_name in REQUIRED_PACKAGES.items()
    if importlib.util.find_spec(module_name) is None
]

if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
    print(f"Installed missing packages: {missing_packages}")
else:
    print("All required packages are already installed.")


## Imports And Config

This next cell imports the notebook dependencies and defines the same core settings used by the midterm notebook, plus the Drive-backed output paths for this setup.

- Reads: Runtime packages, repo paths
- Writes: In-memory constants and config only
- Rerun-safe: Yes. It only defines notebook state.


In [ ]:
import io
import re
import shutil
import xml.etree.ElementTree as ET

import cairosvg
import numpy as np
import pandas as pd
import torch
from IPython.display import display
from PIL import Image
from lxml import etree
from peft import PeftModel
from tqdm import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "Qwen/Qwen2.5-Coder-1.5B-Instruct"
ADAPTER_PATH = CHECKOUT_PATH / "svg-lora-adapter"
MERGED_PATH = CHECKOUT_PATH / "svg-model-merged"
DRIVE_MERGED_PATH = PROJECT_ROOT_ON_DRIVE / "svg-model-merged"

TEST_CSV = CHECKOUT_PATH / "test.csv"
SUBMISSION_CSV = OUTPUT_ROOT / "submission.csv"
DEBUG_CSV = OUTPUT_ROOT / "submission_debug.csv"

MAX_NEW_TOKENS = 1536
BATCH_SIZE = 512
RUN_LIMIT = None  # Set to 10 for a quick smoke test in Colab.

MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256

ALLOWED_TAGS = {
    "svg", "g", "path", "rect", "circle", "ellipse",
    "line", "polyline", "polygon", "defs", "use",
    "symbol", "clipPath", "mask", "linearGradient",
    "radialGradient", "stop", "text", "tspan",
    "title", "desc", "style", "pattern", "marker", "filter"
}

FALLBACK_SVG = (
    '<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 256 256">'
    '<rect width="256" height="256" fill="white"/>'
    '</svg>'
)

torch.manual_seed(1337)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(1337)

print(f"Test CSV: {TEST_CSV}")
print(f"Submission output: {SUBMISSION_CSV}")
print(f"Debug output: {DEBUG_CSV}")
print(f"Run limit: {RUN_LIMIT}")


## Load Model

This next cell follows the midterm-style inference path. It prefers merged model weights, copies them from Drive into the runtime checkout if needed, and only falls back to base model plus LoRA if merged weights are still unavailable.

- Reads: Local merged model directory, Drive merged model directory, adapter files, Hugging Face model files
- Writes: In-memory tokenizer and model state only
- Rerun-safe: Yes. It reloads the model path for the current runtime.


In [ ]:
# TODO: Remove this merged-model autodetect fallback once svg-model-merged has been uploaded to Drive.
if not MERGED_PATH.exists() and DRIVE_MERGED_PATH.exists():
    shutil.copytree(DRIVE_MERGED_PATH, MERGED_PATH)
    print(f"Copied merged model from Drive: {DRIVE_MERGED_PATH}")

if MERGED_PATH.exists():
    tokenizer = AutoTokenizer.from_pretrained(MERGED_PATH)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        MERGED_PATH,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    MODEL_SOURCE = f"merged model at {MERGED_PATH}"
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="auto",
        torch_dtype=torch.float16,
        trust_remote_code=True,
    )
    model = PeftModel.from_pretrained(base_model, str(ADAPTER_PATH))
    MODEL_SOURCE = f"base model + LoRA adapter at {ADAPTER_PATH}"
    print("Merged model not found. Using the temporary base-plus-LoRA fallback.")
    print("Remove this fallback path once svg-model-merged is uploaded to Drive.")

model.eval()
if model.config.pad_token_id is None and tokenizer.pad_token_id is not None:
    model.config.pad_token_id = tokenizer.pad_token_id

print(f"Loaded model source: {MODEL_SOURCE}")
print("pad:", tokenizer.pad_token, tokenizer.pad_token_id)
print("bos:", tokenizer.bos_token, tokenizer.bos_token_id)
print("eos:", tokenizer.eos_token, tokenizer.eos_token_id)
print("model pad:", model.config.pad_token_id)


## SVG Helpers

This next cell ports the permissive midterm SVG extraction, repair, XML recovery, and validation helpers.

- Reads: Raw model text, SVG strings
- Writes: In-memory helper functions only
- Rerun-safe: Yes. It only defines functions.


In [ ]:
def strip_namespace(tag: str) -> str:
    return tag.split("}")[-1] if "}" in tag else tag


def extract_svg(text: str) -> str:
    text = str(text or "").strip()

    match = re.search(r"<svg\b[^>]*>.*?</svg>", text, flags=re.DOTALL | re.IGNORECASE)
    if match:
        return match.group(0).strip()

    if "SVG:" in text:
        text = text.split("SVG:", 1)[1].strip()

    start = text.find("<svg")
    if start != -1:
        return text[start:].strip()

    return text


def render_svg(svg: str, size: int = RENDER_SIZE):
    try:
        png = cairosvg.svg2png(
            bytestring=svg.encode("utf-8"),
            output_width=size,
            output_height=size,
        )
        img = Image.open(io.BytesIO(png)).convert("RGB")
        return np.array(img)
    except Exception:
        return None


def validity_gate(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 0, "svg_too_long_or_empty"

    svg = svg.strip()

    if len(svg) > MAX_SVG_LENGTH:
        return 0, "svg_too_long_or_empty"

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 0, "parse_failed"

    if strip_namespace(root.tag) != "svg":
        return 0, "root_not_svg"

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)

        if tag not in ALLOWED_TAGS:
            return 0, f"disallowed_tag:{tag}"

        if tag == "path":
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 0, "too_many_paths"

    if render_svg(svg) is None:
        return 0, "render_failed"

    return 1, "valid"


def repair_svg(svg: str) -> str:
    if not svg:
        return svg

    svg = svg.strip()

    start = svg.find("<svg")
    if start != -1:
        svg = svg[start:]

    if "SVG:" in svg:
        svg = svg.split("SVG:", 1)[-1].strip()

    if "</svg>" in svg:
        end = svg.rfind("</svg>")
        svg = svg[: end + len("</svg>")]

    if "<svg" in svg and "</svg>" not in svg:
        svg += "</svg>"

    svg = re.sub(r"[A-Za-z0-9.\-]+$", "", svg)
    svg = re.sub(r"<[^>]*$", "", svg)
    return svg


def recover_svg_with_lxml(svg: str) -> str:
    if not svg or "<svg" not in svg:
        return svg
    try:
        parser = etree.XMLParser(recover=True)
        root = etree.fromstring(svg.encode("utf-8"), parser=parser)
        if root is None:
            return svg
        return etree.tostring(root, encoding="unicode")
    except Exception:
        return svg


def looks_collapsed(svg: str) -> bool:
    try:
        root = ET.fromstring(svg)
    except Exception:
        return True

    paths = [elem for elem in root.iter() if strip_namespace(elem.tag) == "path"]
    nonempty_paths = [path for path in paths if path.attrib.get("d", "").strip()]

    if len(paths) > 0 and len(nonempty_paths) == 0:
        return True

    total_elems = sum(1 for _ in root.iter())
    if total_elems <= 1:
        return True

    return False


def accept_repaired_svg(original_svg: str, candidate_svg: str) -> bool:
    valid, _ = validity_gate(candidate_svg)
    if valid == 0:
        return False

    if looks_collapsed(candidate_svg):
        return False

    if len(candidate_svg) < 80:
        return False

    return True


def clean_svg_output(raw_text: str):
    svg = extract_svg(raw_text)

    valid, reason = validity_gate(svg)
    if valid == 1:
        return svg, "valid", svg, raw_text

    repaired = repair_svg(svg)
    valid, reason = validity_gate(repaired)
    if valid == 1:
        return repaired, "repaired", svg, raw_text

    xml = recover_svg_with_lxml(repaired)
    valid, reason = validity_gate(xml)
    if valid == 1:
        return xml, "xml", svg, raw_text

    return FALLBACK_SVG, "fallback", svg, raw_text


## Batched Generation And Submission Build

This next cell ports the midterm notebook’s prompt format, batched generation function, and final submission builder.

- Reads: Test prompts, tokenizer, model, SVG helper functions
- Writes: In-memory generated SVGs plus the final CSV files when `build_submission_csv` runs
- Rerun-safe: Yes. It only defines functions until the final execution cell calls them.


In [ ]:
def build_model_input(prompt: str) -> str:
    return f"Prompt: {prompt}\nSVG:\n"


def generate_svg_batch_debug(prompts, batch_size):
    final_svgs = []
    debug_rows = []

    for i in tqdm(range(0, len(prompts), batch_size)):
        batch_prompts = prompts[i:i + batch_size]
        inputs_text = [build_model_input(prompt) for prompt in batch_prompts]

        inputs = tokenizer(
            inputs_text,
            return_tensors="pt",
            padding=True,
            truncation=True,
        ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id,
            )

        prompt_lengths = inputs["attention_mask"].sum(dim=1).tolist()
        generated_only = [
            outputs[j, int(prompt_lengths[j]):]
            for j in range(outputs.shape[0])
        ]

        decoded = tokenizer.batch_decode(generated_only, skip_special_tokens=True)

        for prompt, raw in zip(batch_prompts, decoded):
            final_svg, reason, extracted_svg, raw_text = clean_svg_output(raw)
            final_svgs.append(final_svg)
            debug_rows.append(
                {
                    "prompt": prompt,
                    "reason": reason,
                    "raw_text": raw_text,
                    "extracted_svg": extracted_svg,
                    "final_svg": final_svg,
                }
            )

    return final_svgs, pd.DataFrame(debug_rows)


def build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    debug_csv_path=DEBUG_CSV,
    batch_size=BATCH_SIZE,
    limit=RUN_LIMIT,
):
    test_df = pd.read_csv(test_csv_path)
    test_df = test_df.dropna(subset=["id", "prompt"]).copy()
    test_df["prompt"] = test_df["prompt"].astype(str)

    if limit is not None:
        test_df = test_df.head(limit).copy()

    prompts = test_df["prompt"].tolist()
    ids = test_df["id"].tolist()

    svgs, debug_df = generate_svg_batch_debug(prompts, batch_size=batch_size)
    print(debug_df["reason"].value_counts())

    assert len(ids) == len(svgs), f"ids={len(ids)}, svgs={len(svgs)}"

    submission_df = pd.DataFrame({
        "id": ids,
        "svg": svgs,
    })

    output_csv_path.parent.mkdir(parents=True, exist_ok=True)
    debug_csv_path.parent.mkdir(parents=True, exist_ok=True)
    submission_df.to_csv(output_csv_path, index=False)
    debug_df.to_csv(debug_csv_path, index=False)
    return submission_df


## Run Submission Build

This next cell runs the full midterm-style submission build and writes only `submission.csv` and `submission_debug.csv` into the persistent Drive output directory.

- Reads: `test.csv`, tokenizer, model, helper functions
- Writes: `submission.csv`, `submission_debug.csv`
- Rerun-safe: Yes. It overwrites the saved output files.


In [ ]:
submission_df = build_submission_csv(
    test_csv_path=TEST_CSV,
    output_csv_path=SUBMISSION_CSV,
    debug_csv_path=DEBUG_CSV,
    batch_size=BATCH_SIZE,
    limit=RUN_LIMIT,
)

print(f"Saved submission to: {SUBMISSION_CSV}")
print(f"Saved debug output to: {DEBUG_CSV}")


## Final Review

This next cell reads the saved output files back from Drive and previews the final submission and debug dataframes.

- Reads: `submission.csv`, `submission_debug.csv`
- Writes: Nothing
- Rerun-safe: Yes. Read-only review cell.


In [ ]:
saved_submission_df = pd.read_csv(SUBMISSION_CSV, keep_default_na=False)
saved_debug_df = pd.read_csv(DEBUG_CSV, keep_default_na=False)

print(f"Saved submission rows: {len(saved_submission_df)}")
print(f"Saved debug rows: {len(saved_debug_df)}")
display(saved_submission_df.head())
display(saved_debug_df.head())
